**Модель:** игра с семью игроками  
**Начальный баланс:** 20 монет у каждого игрока  
**Количество раундов:** 20  
**Вероятности выигрыша:** у игроков с четными номерами вес выигрыша в два раза выше  
**Хранение переводов:** блокчейн с Proof of Work

В каждом раунде проигравшие переводят победителю по одной монете.
Игрок с минимальным балансом освобождается от перевода.

## 0. Импорт библиотек

In [ ]:
import hashlib
import json
import random
import time

import pandas as pd
import matplotlib.pyplot as plt

## 1. Параметры игры

In [ ]:
SEED = 42
random.seed(SEED)

N = 7
INITIAL_BALANCE = 20
ROUNDS_COUNT = 20

weights = [
    1 if player % 2 == 1 else 2
    for player in range(1, N + 1)
]

total_weight = sum(weights)

probabilities = [
    weight / total_weight
    for weight in weights
]

players = {
    player: INITIAL_BALANCE
    for player in range(1, N + 1)
}

probabilities_df = pd.DataFrame({
    "Игрок": range(1, N + 1),
    "Вес": weights,
    "Вероятность": probabilities,
})

display(probabilities_df)
print("Сумма вероятностей:", sum(probabilities))

## 2. Блок и цепочка блоков

In [ ]:
class Block:
    def __init__(
        self,
        index,
        transactions,
        previous_hash,
        timestamp=None,
        nonce=0,
    ):
        self.index = index
        self.timestamp = (
            time.time()
            if timestamp is None
            else timestamp
        )
        self.transactions = list(transactions)
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        payload = {
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce,
        }

        block_text = json.dumps(
            payload,
            sort_keys=True,
            ensure_ascii=False
        )

        return hashlib.sha256(
            block_text.encode("utf-8")
        ).hexdigest()


class Blockchain:
    def __init__(self, difficulty=4, block_size=5):
        self.difficulty = difficulty
        self.block_size = block_size
        self.chain = []
        self.current_transactions = []
        self.add_genesis_block()

    def add_genesis_block(self):
        genesis = Block(
            index=0,
            transactions=["Genesis Block"],
            previous_hash="0",
        )
        self.chain.append(genesis)

    def add_transaction(self, transaction):
        self.current_transactions.append(transaction)

        if len(self.current_transactions) >= self.block_size:
            self.mine_current_transactions()

    def mine_current_transactions(self):
        if not self.current_transactions:
            return None

        previous_hash = self.chain[-1].hash

        block = Block(
            index=len(self.chain),
            transactions=self.current_transactions,
            previous_hash=previous_hash,
        )

        prefix = "0" * self.difficulty

        while not block.hash.startswith(prefix):
            block.nonce += 1
            block.hash = block.calculate_hash()

        self.chain.append(block)
        self.current_transactions = []

        return block

    def is_valid(self):
        for index in range(1, len(self.chain)):
            current = self.chain[index]
            previous = self.chain[index - 1]

            if current.previous_hash != previous.hash:
                return False

            if current.hash != current.calculate_hash():
                return False

            if not current.hash.startswith(
                "0" * self.difficulty
            ):
                return False

        return True

## 3. Симуляция

In [ ]:
def run_simulation():
    balances = {
        player: INITIAL_BALANCE
        for player in range(1, N + 1)
    }

    blockchain = Blockchain(
        difficulty=4,
        block_size=5
    )

    round_results = []
    round_transactions = []

    for round_number in range(1, ROUNDS_COUNT + 1):
        winner = random.choices(
            list(balances.keys()),
            weights=probabilities,
            k=1
        )[0]

        min_player = min(
            balances,
            key=lambda player: balances[player]
        )

        before = balances.copy()

        transaction_row = {
            player: 0
            for player in balances
        }

        for player in balances:
            if player == winner:
                continue

            if player == min_player:
                continue

            if balances[player] <= 0:
                continue

            balances[player] -= 1
            balances[winner] += 1

            transaction = {
                "round": round_number,
                "from": player,
                "to": winner,
                "amount": 1,
            }

            blockchain.add_transaction(
                transaction
            )

            transaction_row[player] -= 1
            transaction_row[winner] += 1

        round_results.append({
            "Раунд": round_number,
            "Победитель": winner,
            "Баланс победителя": balances[winner],
            "Освобожден от перевода": min_player,
            "Баланс до": before[winner],
        })

        transaction_row["Раунд"] = round_number
        round_transactions.append(transaction_row)

    if blockchain.current_transactions:
        blockchain.mine_current_transactions()

    return (
        balances,
        round_results,
        round_transactions,
        blockchain,
    )


(
    final_balances,
    round_results,
    round_transactions,
    blockchain
) = run_simulation()

## 4. Результаты раундов

In [ ]:
rounds_df = pd.DataFrame(round_results)
display(rounds_df)

## 5. Транзакции по игрокам

In [ ]:
transactions_df = pd.DataFrame(
    round_transactions
)

columns = ["Раунд"] + list(range(1, N + 1))
transactions_df = transactions_df[columns]

display(transactions_df)

## 6. Итоговые балансы

In [ ]:
balances_df = pd.DataFrame({
    "Игрок": list(final_balances.keys()),
    "Баланс": list(final_balances.values()),
})

display(balances_df)

print(
    "Сумма монет:",
    sum(final_balances.values())
)

print(
    "Начальная сумма:",
    N * INITIAL_BALANCE
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    balances_df["Игрок"],
    balances_df["Баланс"]
)
plt.xlabel("Игрок")
plt.ylabel("Количество монет")
plt.title("Итоговые балансы игроков")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 7. Содержимое блокчейна

In [ ]:
chain_rows = []

for block in blockchain.chain:
    chain_rows.append({
        "Индекс": block.index,
        "Транзакций": len(block.transactions),
        "Nonce": block.nonce,
        "Hash": block.hash[:20] + "...",
        "Previous hash": block.previous_hash[:20] + "...",
    })

chain_df = pd.DataFrame(chain_rows)

display(chain_df)

print("Количество блоков:", len(blockchain.chain))
print("Цепочка корректна:", blockchain.is_valid())

## 8. Частота побед

In [ ]:
winner_counts = (
    rounds_df["Победитель"]
    .value_counts()
    .sort_index()
)

winner_stats = pd.DataFrame({
    "Игрок": range(1, N + 1),
    "Побед": [
        int(winner_counts.get(player, 0))
        for player in range(1, N + 1)
    ],
    "Теоретическая вероятность": probabilities,
})

display(winner_stats)

## Итог

In [ ]:
print("Раундов:", ROUNDS_COUNT)
print("Блоков:", len(blockchain.chain))
print("Цепочка корректна:", blockchain.is_valid())
print("Сумма монет сохранена:", sum(final_balances.values()) == N * INITIAL_BALANCE)